# Python Week-2
## 建立CoLabNotebook
## 執行程式


In [4]:
course_name = "Python與資料科學"

print("歡迎來到:",course_name)

歡迎來到: Python與資料科學


In [ ]:
a=10
b=11
print(a+b)

21


In [ ]:
stray= 'hello world~~'
print(stray)

hello world~~


In [1]:
# ============================================================
#  Google Colab 實用功能示範
#  執行方式：複製整段 -> 貼到 Colab 新儲存格 -> Shift+Enter
# ============================================================

# ------------------------------------------------------------
# 1️⃣ 環境檢查：確認是否有 GPU 可用
# ------------------------------------------------------------
import torch, tensorflow as tf, sys, subprocess, json, os, textwrap, math, random, datetime

print("🐍 Python 版本 :", sys.version.split()[0])
print("🔧 PyTorch 版本 :", torch.__version__)
print("🔧 TensorFlow 版本 :", tf.__version__)

# 檢查 GPU
gpu_info = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print("✅ 偵測到 GPU：")
    for line in gpu_info.stdout.strip().split("\n"):
        print("   -", line)
    # PyTorch / TF 也確認一下
    print("   PyTorch cuda 可用 :", torch.cuda.is_available())
    print("   TF GPU 裝置      :", tf.config.list_physical_devices('GPU'))
else:
    print("⚠️ 未偵測到 GPU（執行階段 → 變更執行階段類型 → 選 GPU 可啟用）")

print("\n" + "="*60)

# ------------------------------------------------------------
# 2️⃣ 視覺化：畫一個漂亮的數學函數動畫圖
# ------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

x = np.linspace(-2*np.pi, 2*np.pi, 400)
y1 = np.sin(x)
y2 = np.cos(x)
y3 = np.sin(x) * np.exp(-0.1*np.abs(x))  # 振幅遞減的正弦波

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_title("Math Functions Visualization", fontsize=14)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(True, alpha=0.3)
ax.set_xlim(-2*np.pi, 2*np.pi)
ax.set_ylim(-1.5, 1.5)

line1, = ax.plot([], [], label="sin(x)", color="#1f77b4", linewidth=2)
line2, = ax.plot([], [], label="cos(x)", color="#ff7f0e", linewidth=2, linestyle="--")
line3, = ax.plot([], [], label="sin(x)·e^(-0.1|x|)", color="#2ca02c", linewidth=2, linestyle=":")
ax.legend(loc="upper right")

def init():
    for line in (line1, line2, line3):
        line.set_data([], [])
    return line1, line2, line3

def animate(i):
    # 逐步繪製
    idx = min(i*5, len(x))
    line1.set_data(x[:idx], y1[:idx])
    line2.set_data(x[:idx], y2[:idx])
    line3.set_data(x[:idx], y3[:idx])
    return line1, line2, line3

ani = animation.FuncAnimation(fig, animate, frames=80, init_func=init,
                              interval=50, blit=True, repeat=False)
plt.close()  # 避免靜態圖重複顯示
display(HTML(ani.to_jshtml()))  # 在 Colab 直接播放動畫

print("="*60)

# ------------------------------------------------------------
# 3️⃣ 真實資料：從 UCI 下載「紅酒品質」資料集並做 EDA
# ------------------------------------------------------------
import pandas as pd
import seaborn as sns

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=";")

print("📊 資料集形狀 :", df.shape)
print("\n📋 前 5 筆 :")
display(df.head())

print("\n📈 基本統計 :")
display(df.describe().T.style.background_gradient(cmap="Blues"))

# 相關係數熱力圖
plt.figure(figsize=(10, 8))
arr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(arr, dtype=bool))
sns.heatmap(arr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Feature Correlation Heatmap (Red Wine Quality)", fontsize=14)
plt.tight_layout()
plt.show()

print("="*60)

# ------------------------------------------------------------
# 4️⃣ 快速建模：Random Forest 預測酒品質（分類）
# ------------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# 將 quality 轉為三類：低(3-5)、中(6)、高(7-8)
def quality_to_class(q):
    if q <= 5: return 0  # Low
    elif q == 6: return 1  # Medium
    else: return 2  # High

df["quality_class"] = df["quality"].apply(quality_to_class)

X = df.drop(["quality", "quality_class"], axis=1)
y = df["quality_class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=200, max_depth=12,
                             random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("🎯 分類報告 :")
print(classification_report(y_test, y_pred,
                            target_names=["Low (≤5)", "Medium (6)", "High (≥7)"]))

# 混淆矩陣
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
                                        display_labels=["Low", "Medium", "High"],
                                        cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

# 特徵重要性
importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(6, 5))
importances.plot(kind="barh", color="#1f77b4", edgecolor="black")
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("\n✅ 所有示範完成！你可以繼續在下方儲存格撰寫自己的程式碼。")

SyntaxError: positional argument follows keyword argument (127184206.py, line 52)